# 🔥 CalorieSense — Calorie Burn Prediction Pipeline

**AI Project (I) — National Institute of Business Management**

This notebook implements the full ML pipeline for predicting calories burned during physical activity, following the methodology outlined in the project proposal:

1. Data Acquisition & Loading
2. Data Cleaning & Quality Checks
3. Exploratory Data Analysis (EDA)
4. Feature Preparation & Engineering
5. Model Training & Comparison
6. Model Evaluation
7. Save Best Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost
from xgboost import XGBRegressor

# ANN (TensorFlow / Keras)
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    USE_KERAS = True
except ImportError:
    from sklearn.neural_network import MLPRegressor
    USE_KERAS = False
    print("⚠️ TensorFlow not found — will use sklearn MLPRegressor for ANN")

# Plot settings
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Create output directory
os.makedirs('out', exist_ok=True)

print("✅ All libraries imported successfully")

---
## 📂 1. Data Acquisition & Loading

The project uses **4 datasets** as described in the proposal:

| # | Dataset | Role |
|---|---------|------|
| 1 | Gym Members Exercise Tracking | **Primary** — calorie prediction training |
| 2 | Exercise & Fitness Metrics | **Supplement** — adds weather & intensity context |
| 3 | Fitness Exercises Library | **Lookup** — runtime exercise recommendations (not for training) |
| 4 | Mendeley Gym Recommendations | **Lookup** — AI coaching at runtime (not for training) |

In [ ]:
# Load all 4 datasets
df1 = pd.read_csv('data/1. Gym Members Exercise Dataset/gym_members_exercise_tracking.csv')
df2 = pd.read_csv('data/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')
df3 = pd.read_csv('data/3. Fitness Exercises Dataset/exercises_flat.csv')
df4 = pd.read_excel('data/4. Mendeley Gym Recommendation Dataset/gym recommendation.xlsx')

print(f"Dataset 1 — Gym Members:          {df1.shape[0]:>6,} rows × {df1.shape[1]} cols")
print(f"Dataset 2 — Exercise Metrics:      {df2.shape[0]:>6,} rows × {df2.shape[1]} cols")
print(f"Dataset 3 — Exercise Library:      {df3.shape[0]:>6,} rows × {df3.shape[1]} cols  (lookup only)")
print(f"Dataset 4 — Gym Recommendations:   {df4.shape[0]:>6,} rows × {df4.shape[1]} cols  (lookup only)")

---
## 🔍 2. Data Inspection

### 2.1 Dataset 1 — Gym Members Exercise Tracking (Primary)

In [ ]:
print(f"Shape: {df1.shape}")
print(f"\nColumns: {df1.columns.tolist()}")
print(f"\nData Types:")
print(df1.dtypes)
print(f"\nFirst 5 rows:")
df1.head()

### 2.2 Dataset 2 — Exercise & Fitness Metrics (Supplement)

In [ ]:
print(f"Shape: {df2.shape}")
print(f"\nColumns: {df2.columns.tolist()}")
print(f"\nData Types:")
print(df2.dtypes)
print(f"\nFirst 5 rows:")
df2.head()

### 2.3 Datasets 3 & 4 — Lookup Tables (Runtime Only)

> **Note:** These datasets are NOT used for model training. Dataset 3 provides exercise recommendations and Dataset 4 provides AI coaching — both queried at runtime in the web application.

In [ ]:
print("=== Dataset 3: Fitness Exercises Library (30 exercises) ===")
print(f"Columns: {df3.columns.tolist()}")
print(f"Target muscles: {df3['target_muscles'].unique().tolist()}")
print(f"Body parts: {df3['body_part_category'].unique().tolist()}")
print(f"Equipment: {df3['equipment_needed'].unique().tolist()}")

print(f"\n=== Dataset 4: Gym Recommendations ({df4.shape[0]:,} entries) ===")
print(f"Columns: {df4.columns.tolist()}")
print(f"BMI Levels: {df4['Level'].unique().tolist()}")
print(f"Fitness Goals: {df4['Fitness Goal'].unique().tolist()}")
print(f"Fitness Types: {df4['Fitness Type'].unique().tolist()}")

---
## 🧹 3. Data Cleaning & Quality Checks

In [ ]:
# --- Missing Values ---
print("=" * 50)
print("MISSING VALUES CHECK")
print("=" * 50)
for label, df in [("Dataset 1 (Gym Members)", df1), ("Dataset 2 (Exercise Metrics)", df2)]:
    missing = df.isnull().sum()
    has_missing = missing[missing > 0]
    if len(has_missing):
        print(f"\n⚠️ {label}:")
        print(has_missing)
    else:
        print(f"✅ {label}: No missing values")

# --- Duplicates ---
print(f"\n{'=' * 50}")
print("DUPLICATE ROWS CHECK")
print("=" * 50)
d1_dup = df1.duplicated().sum()
d2_dup = df2.duplicated().sum()
print(f"Dataset 1 duplicates: {d1_dup} {'✅' if d1_dup == 0 else '⚠️'}")
print(f"Dataset 2 duplicates: {d2_dup} {'✅' if d2_dup == 0 else '⚠️'}")

# Remove duplicates if any
if d1_dup > 0:
    df1 = df1.drop_duplicates()
    print(f"  → Removed {d1_dup} duplicates from Dataset 1")
if d2_dup > 0:
    df2 = df2.drop_duplicates()
    print(f"  → Removed {d2_dup} duplicates from Dataset 2")

---
## 📊 4. Descriptive Statistics

In [ ]:
print("=== Dataset 1: Descriptive Statistics ===")
df1.describe().round(2)

In [ ]:
print("=== Dataset 2: Descriptive Statistics ===")
df2.describe().round(2)

---
## ⚙️ 5. Data Preprocessing

**Merge Strategy:** Combine Datasets 1 & 2 on their common features to build a larger, unified training set.

| Dataset 1 Column | Dataset 2 Column | Unified Name |
|---|---|---|
| Age | Age | Age |
| Gender | Gender | Gender |
| Weight (kg) | Actual Weight | Weight |
| BMI | BMI | BMI |
| Session_Duration (hours) × 60 | Duration (minutes) | Duration_Min |
| Avg_BPM | Heart Rate | Heart_Rate |
| Calories_Burned | Calories Burn | **Calories** (target) |

In [ ]:
# ─── Prepare Dataset 1 ───
df1_clean = df1.copy()
df1_clean = df1_clean.rename(columns={
    'Weight (kg)': 'Weight',
    'Height (m)': 'Height',
    'Avg_BPM': 'Heart_Rate',
    'Session_Duration (hours)': 'Duration_Min',
    'Calories_Burned': 'Calories'
})

# Convert duration from hours → minutes
df1_clean['Duration_Min'] = (df1_clean['Duration_Min'] * 60).round(1)

# Select common features
df1_prep = df1_clean[['Age', 'Gender', 'Weight', 'BMI',
                       'Duration_Min', 'Heart_Rate', 'Calories']].copy()
df1_prep['Source'] = 'DS1'

print(f"✅ Dataset 1 prepared: {df1_prep.shape}")
df1_prep.head()

In [ ]:
# ─── Prepare Dataset 2 ───
df2_clean = df2.copy()
df2_clean = df2_clean.rename(columns={
    'Actual Weight': 'Weight',
    'Heart Rate': 'Heart_Rate',
    'Duration': 'Duration_Min',
    'Calories Burn': 'Calories',
    'Weather Conditions': 'Weather',
    'Exercise Intensity': 'Intensity'
})

# Select common features
df2_prep = df2_clean[['Age', 'Gender', 'Weight', 'BMI',
                       'Duration_Min', 'Heart_Rate', 'Calories']].copy()
df2_prep['Source'] = 'DS2'

print(f"✅ Dataset 2 prepared: {df2_prep.shape}")
df2_prep.head()

In [ ]:
# ─── Merge into unified training set ───
df_merged = pd.concat([df1_prep, df2_prep], ignore_index=True)

print(f"✅ Merged dataset: {df_merged.shape[0]:,} rows × {df_merged.shape[1]} cols")
print(f"   From DS1: {(df_merged['Source'] == 'DS1').sum():,} rows")
print(f"   From DS2: {(df_merged['Source'] == 'DS2').sum():,} rows")
print(f"\nMissing values after merge:")
print(df_merged.isnull().sum())
df_merged.head()

In [ ]:
# ─── Encode categorical variables ───
le_gender = LabelEncoder()
df_merged['Gender_Encoded'] = le_gender.fit_transform(df_merged['Gender'])
print(f"Gender mapping: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}")

# Final training dataframe (drop non-numeric columns)
df_train = df_merged.drop(columns=['Gender', 'Source']).copy()

print(f"\n✅ Final training dataset: {df_train.shape}")
print(f"Columns: {df_train.columns.tolist()}")
df_train.head()

---
## 📈 6. Exploratory Data Analysis (EDA)

### 6.1 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_train.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('out/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Feature Distributions

In [ ]:
features_to_plot = ['Age', 'Weight', 'BMI', 'Duration_Min', 'Heart_Rate', 'Calories']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    axes[i].hist(df_train[feat], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_title(f'{feat}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Frequency')
    mean_val = df_train[feat].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_val:.1f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Scatter Plots — Key Features vs Calories

In [ ]:
key_features = ['Duration_Min', 'Heart_Rate', 'Weight', 'BMI']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    ax.scatter(df_train[feat], df_train['Calories'], alpha=0.3, s=10, color='teal')
    ax.set_xlabel(feat, fontsize=12)
    ax.set_ylabel('Calories Burned', fontsize=12)

    # Trend line
    z = np.polyfit(df_train[feat], df_train['Calories'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_train[feat].min(), df_train[feat].max(), 100)
    r = df_train[feat].corr(df_train['Calories'])
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'r = {r:.2f}')
    ax.set_title(f'{feat} vs Calories', fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle('Key Features vs Calories Burned', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Box Plots — Calories by Gender & Source

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_merged, x='Gender', y='Calories', ax=axes[0], palette='Set2')
axes[0].set_title('Calories Burned by Gender', fontsize=13, fontweight='bold')

sns.boxplot(data=df_merged, x='Source', y='Calories', ax=axes[1], palette='Set3')
axes[1].set_title('Calories Burned by Dataset Source', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('out/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Pairwise Feature Comparison

In [ ]:
sample = df_train.sample(n=min(1000, len(df_train)), random_state=42)
pair = sns.pairplot(sample[['Age', 'Weight', 'Duration_Min', 'Heart_Rate', 'Calories']],
                    diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
pair.fig.suptitle('Pairwise Feature Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('out/pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 🎯 7. Feature Importance Analysis

In [ ]:
X_fi = df_train.drop(columns=['Calories'])
y_fi = df_train['Calories']

rf_fi = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_fi.fit(X_fi, y_fi)

importance = pd.DataFrame({
    'Feature': X_fi.columns,
    'Importance': rf_fi.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(importance['Feature'], importance['Importance'],
                color='steelblue', edgecolor='black')
plt.xlabel('Importance Score', fontsize=12)
plt.title('Feature Importance (Random Forest)', fontsize=16, fontweight='bold')
for i, (val, name) in enumerate(zip(importance['Importance'], importance['Feature'])):
    plt.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('out/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFeature Importance Ranking:")
for _, row in importance.sort_values('Importance', ascending=False).iterrows():
    print(f"  {row['Feature']:20s} → {row['Importance']:.4f}")

---
## ✂️ 8. Train / Test Split & Feature Scaling

In [ ]:
# Define features (X) and target (y)
X = df_train.drop(columns=['Calories'])
y = df_train['Calories']

print(f"Features ({X.shape[1]}): {X.columns.tolist()}")
print(f"Target: Calories")
print(f"Total samples: {X.shape[0]:,}")

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")

# Feature scaling (needed for Linear Regression and ANN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ StandardScaler fitted on training data")

---
## 🤖 9. Model Training

Training **4 models** as specified in the proposal:
1. Linear Regression
2. Random Forest Regression
3. Gradient Boosting (XGBoost)
4. Artificial Neural Network (ANN)

### 9.1 Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2   = r2_score(y_test, lr_pred)

print("Linear Regression Results:")
print(f"  MAE:  {lr_mae:.2f}")
print(f"  RMSE: {lr_rmse:.2f}")
print(f"  R²:   {lr_r2:.4f}")

### 9.2 Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2   = r2_score(y_test, rf_pred)

print("Random Forest Results:")
print(f"  MAE:  {rf_mae:.2f}")
print(f"  RMSE: {rf_rmse:.2f}")
print(f"  R²:   {rf_r2:.4f}")

### 9.3 Gradient Boosting (XGBoost)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2   = r2_score(y_test, xgb_pred)

print("XGBoost Results:")
print(f"  MAE:  {xgb_mae:.2f}")
print(f"  RMSE: {xgb_rmse:.2f}")
print(f"  R²:   {xgb_r2:.4f}")

### 9.4 Artificial Neural Network (ANN)

In [ ]:
if USE_KERAS:
    # ─── Keras Sequential Model ───
    ann_model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')
    ])

    ann_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    ann_model.summary()

    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

    history = ann_model.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=150,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )
    ann_pred = ann_model.predict(X_test_scaled).flatten()
else:
    # ─── Sklearn MLPRegressor fallback ───
    ann_model = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32, 16),
        activation='relu',
        solver='adam',
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.2,
        random_state=42,
        verbose=True
    )
    ann_model.fit(X_train_scaled, y_train)
    ann_pred = ann_model.predict(X_test_scaled)
    history = None

ann_mae  = mean_absolute_error(y_test, ann_pred)
ann_rmse = np.sqrt(mean_squared_error(y_test, ann_pred))
ann_r2   = r2_score(y_test, ann_pred)

print(f"\nANN Results:")
print(f"  MAE:  {ann_mae:.2f}")
print(f"  RMSE: {ann_rmse:.2f}")
print(f"  R²:   {ann_r2:.4f}")

In [ ]:
# ─── ANN Training History (Keras only) ───
if USE_KERAS and history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0].set_title('ANN Training Loss (MSE)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(history.history['mae'], label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Validation MAE')
    axes[1].set_title('ANN Training MAE', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('out/ann_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Training history plot not available (sklearn backend)")

---
## 📋 10. Model Evaluation & Comparison

### 10.1 Performance Comparison Table

In [ ]:
# Build comparison table
results = {
    'Linear Regression': {'predictions': lr_pred, 'MAE': lr_mae, 'RMSE': lr_rmse, 'R2': lr_r2},
    'Random Forest':     {'predictions': rf_pred, 'MAE': rf_mae, 'RMSE': rf_rmse, 'R2': rf_r2},
    'XGBoost':           {'predictions': xgb_pred,'MAE': xgb_mae,'RMSE': xgb_rmse,'R2': xgb_r2},
    'ANN':               {'predictions': ann_pred,'MAE': ann_mae,'RMSE': ann_rmse,'R2': ann_r2},
}

comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [r['MAE'] for r in results.values()],
    'RMSE': [r['RMSE'] for r in results.values()],
    'R² Score': [r['R2'] for r in results.values()]
}).sort_values('R² Score', ascending=False).reset_index(drop=True)

print("=" * 60)
print("           MODEL COMPARISON RESULTS")
print("=" * 60)
print(comparison.to_string(index=False))

best_model_name = comparison.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_model_name}  (R² = {comparison.iloc[0]['R² Score']:.4f})")

### 10.2 Comparison Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = comparison['Model'].tolist()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

metrics_data = [
    ('MAE', 'MAE (Lower is Better)'),
    ('RMSE', 'RMSE (Lower is Better)'),
    ('R² Score', 'R² Score (Higher is Better)')
]

for ax, (col, title) in zip(axes, metrics_data):
    bars = ax.bar(models, comparison[col], color=colors[:len(models)], edgecolor='black')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(col)
    for bar, val in zip(bars, comparison[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.3 Cross-Validation (5-Fold)

In [ ]:
print("=" * 60)
print("       5-FOLD CROSS-VALIDATION (R² Score)")
print("=" * 60)

cv_models = {
    'Linear Regression': (LinearRegression(), True),
    'Random Forest': (RandomForestRegressor(n_estimators=200, max_depth=15,
                      random_state=42, n_jobs=-1), False),
    'XGBoost': (XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.1,
                random_state=42, verbosity=0), False),
}

cv_results = {}
for name, (model, needs_scaling) in cv_models.items():
    X_cv = X_train_scaled if needs_scaling else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=5, scoring='r2')
    cv_results[name] = scores
    print(f"{name:25s} → Mean R²: {scores.mean():.4f} ± {scores.std():.4f}")

print("\n(ANN cross-validation skipped — computationally expensive; uses validation split)")

### 10.4 Actual vs Predicted Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    ax = axes[i]
    ax.scatter(y_test, res['predictions'], alpha=0.4, s=15, color='teal')

    min_val = min(y_test.min(), res['predictions'].min())
    max_val = max(y_test.max(), res['predictions'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2,
            label='Perfect Prediction')

    ax.set_xlabel('Actual Calories', fontsize=11)
    ax.set_ylabel('Predicted Calories', fontsize=11)
    ax.set_title(f'{name}\nR² = {res["R2"]:.4f} | MAE = {res["MAE"]:.1f}',
                 fontsize=12, fontweight='bold')
    ax.legend()

plt.suptitle('Actual vs Predicted Calories', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.5 Residual Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    ax = axes[i]
    residuals = y_test.values - res['predictions']
    ax.scatter(res['predictions'], residuals, alpha=0.4, s=15, color='coral')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Predicted Calories', fontsize=11)
    ax.set_ylabel('Residuals', fontsize=11)
    ax.set_title(f'{name} Residuals', fontsize=12, fontweight='bold')

plt.suptitle('Residual Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/residual_plots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 11. Save Best Model & Artifacts

In [ ]:
import joblib

# Map model names to model objects
model_objects = {
    'Linear Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'ANN': ann_model,
}

best_name = comparison.iloc[0]['Model']
best_model = model_objects[best_name]

print(f"🏆 Saving best model: {best_name}")
print(f"   R² = {comparison.iloc[0]['R² Score']:.4f}")

os.makedirs('out', exist_ok=True)

# Save model
if best_name == 'ANN' and USE_KERAS:
    best_model.save('out/best_model_ann.keras')
    print("   → Saved: out/best_model_ann.keras")
else:
    joblib.dump(best_model, 'out/best_model.pkl')
    print("   → Saved: out/best_model.pkl")

# Save preprocessing artifacts
joblib.dump(scaler, 'out/scaler.pkl')
print("   → Saved: out/scaler.pkl")

joblib.dump(le_gender, 'out/le_gender.pkl')
print("   → Saved: out/le_gender.pkl")

feature_names = X.columns.tolist()
joblib.dump(feature_names, 'out/feature_names.pkl')
print(f"   → Saved: out/feature_names.pkl")
print(f"   → Features: {feature_names}")

# Save comparison results
comparison.to_csv('out/model_comparison.csv', index=False)
print("   → Saved: out/model_comparison.csv")

print(f"\n✅ All artifacts saved to out/ directory")

---
## 📝 Summary & Next Steps

### ✅ Completed in this notebook:
1. Loaded and inspected all 4 datasets
2. Data quality checks — no missing values, duplicates handled
3. Preprocessed and merged Datasets 1 & 2 into a unified training set
4. Comprehensive EDA with correlation heatmap, distributions, scatter plots, box plots, pairplot
5. Feature importance analysis
6. Trained 4 models: Linear Regression, Random Forest, XGBoost, ANN
7. Evaluated with MAE, RMSE, R², cross-validation, actual vs predicted, residuals
8. Saved best model and all preprocessing artifacts to `out/`

### 🔜 Next Steps — Web Application:
- Build web app with user input form for calorie prediction
- Load saved model from `out/` for inference
- Integrate OpenWeather API for live weather context
- Query Dataset 3 for personalized exercise recommendations
- Query Dataset 4 for AI coaching (diet, equipment, exercises)
- Add visual fitness insight charts